
#### refunder agent

this notebook creates an agent with tools to suggest refunds for orders

#### Tool & View Registration

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ${CATALOG}.ai;

In [ ]:
%sql
CREATE OR REPLACE FUNCTION ${CATALOG}.ai.get_order_details(oid STRING COMMENT 'order id of the order')
RETURNS TABLE (
  body STRING COMMENT 'Body of the event',
  event_type STRING COMMENT 'The type of event',
  order_id STRING COMMENT 'The order id',
  ts STRING COMMENT 'The timestamp of the event',
  location STRING COMMENT 'the location of the order'
)
COMMENT 'Returns all events associated with the order id (oid)'
RETURN
  SELECT ae.body, ae.event_type, ae.order_id, ae.ts, loc.name as location
  FROM ${CATALOG}.lakeflow.all_events ae
  LEFT JOIN ${CATALOG}.simulator.locations loc ON ae.location_id = loc.location_id
  WHERE ae.order_id = oid;

In [0]:
%sql
CREATE OR REPLACE FUNCTION ${CATALOG}.ai.get_order_delivery_time(oid STRING COMMENT 'order id of the order')
RETURNS TABLE (
  order_id STRING COMMENT 'The order id',
  creation_time TIMESTAMP COMMENT 'The timestamp of the first event for the order',
  delivery_time TIMESTAMP COMMENT 'The timestamp of the last event for the order',
  duration_minutes FLOAT COMMENT 'The total duration from the first to the last event in minutes'
)
COMMENT 'Returns the first event time, last event time, and total duration for a given order id.'
RETURN
  WITH MinMaxTimestamps AS (
    SELECT
      MIN(try_to_timestamp(ts)) as first_event_time,
      MAX(try_to_timestamp(ts)) as last_event_time
    FROM
      ${CATALOG}.lakeflow.all_events
    WHERE
      order_id = oid
  )
  SELECT
    oid as order_id,
    first_event_time AS creation_time,
    last_event_time AS delivery_time,
    CAST(
      try_divide(
        (UNIX_TIMESTAMP(last_event_time) - UNIX_TIMESTAMP(first_event_time)),
        60
      ) AS FLOAT
    ) AS duration_minutes
  FROM
    MinMaxTimestamps;

In [ ]:
%sql
CREATE OR REPLACE VIEW ${CATALOG}.ai.order_delivery_times_per_location_view AS
WITH order_times AS (
  SELECT
    ae.order_id,
    loc.name as location,
    MAX(CASE WHEN ae.event_type = 'order_created' THEN try_to_timestamp(ae.ts) END) AS order_created_time,
    MAX(CASE WHEN ae.event_type = 'delivered' THEN try_to_timestamp(ae.ts) END) AS delivered_time
  FROM
    ${CATALOG}.lakeflow.all_events ae
  LEFT JOIN ${CATALOG}.simulator.locations loc ON ae.location_id = loc.location_id
  WHERE
    try_to_timestamp(ae.ts) >= CURRENT_TIMESTAMP() - INTERVAL 1 DAY
  GROUP BY
    ae.order_id,
    loc.name
),
total_order_times AS (
  SELECT
    order_id,
    location,
    (UNIX_TIMESTAMP(delivered_time) - UNIX_TIMESTAMP(order_created_time)) / 60 AS total_order_time_minutes
  FROM
    order_times
  WHERE
    order_created_time IS NOT NULL
    AND delivered_time IS NOT NULL
)
SELECT
  location,
  PERCENTILE(total_order_time_minutes, 0.50) AS P50,
  PERCENTILE(total_order_time_minutes, 0.75) AS P75,
  PERCENTILE(total_order_time_minutes, 0.99) AS P99
FROM
  total_order_times
GROUP BY
  location

In [0]:
%sql
CREATE OR REPLACE FUNCTION ${CATALOG}.ai.get_location_timings(loc STRING COMMENT 'Location name as a string')
RETURNS TABLE (
  location STRING COMMENT 'Location of the order source',
  P50 FLOAT COMMENT '50th percentile',
  P75 FLOAT COMMENT '75th percentile',
  P99 FLOAT COMMENT '99th percentile'
)
COMMENT 'Returns the 50/75/99th percentile of total delivery times for locations'
RETURN
  SELECT location, P50, P75, P99
  FROM ${CATALOG}.ai.order_delivery_times_per_location_view AS odlt
  WHERE odlt.location = loc;

In [ ]:
%sql
-- USE CATALOG is needed in addition to USE SCHEMA + EXECUTE so the
-- Databricks App service principal can traverse the catalog to reach the
-- UC functions at inference time.
GRANT USE CATALOG ON CATALOG ${CATALOG} TO `account users`;
GRANT USE SCHEMA ON SCHEMA ${CATALOG}.ai TO `account users`;


In [ ]:
%sql
-- Grant EXECUTE so app callers and the agent app SP can call these tools at inference time.
GRANT EXECUTE ON FUNCTION ${CATALOG}.ai.get_order_details        TO `account users`;
GRANT EXECUTE ON FUNCTION ${CATALOG}.ai.get_order_delivery_time  TO `account users`;
GRANT EXECUTE ON FUNCTION ${CATALOG}.ai.get_location_timings     TO `account users`;


#### App Agent

- Install orchestration dependencies and restart Python for a clean runtime.
- Capture widget inputs (`CATALOG`, `LLM_MODEL`) and resolve the deterministic Databricks App name.
- Use `../apps/refund-agent` as the source of truth for the LangGraph refund workflow.
- Treat `LLM_MODEL` as a Unity AI Gateway endpoint name; no custom-agent LLM calls use legacy model-serving invocation routes.


In [0]:
%pip install -U -qqqq mlflow[databricks] databricks-sdk requests openai
dbutils.library.restartPython()


In [0]:
CATALOG = dbutils.widgets.get("CATALOG")
LLM_MODEL = dbutils.widgets.get("LLM_MODEL")

import sys
sys.path.append('../utils')
from agent_app_client import refund_agent_app_name

APP_NAME = refund_agent_app_name(CATALOG)
UC_MODEL_NAME = f"{CATALOG}.ai.refund_agent_app"
print(f"Refund agent app: {APP_NAME}")


In [ ]:
import mlflow

# Create/set dev experiment for development and evaluation traces.
# Use a shared path so the job-runner SP and the deployer both write to the
# same experiment and the runbook can link to a stable URL.  Mirrors the
# pattern in stages/complaint_agent.ipynb so all three pipeline agents
# (refund / complaint / supervisor) share the same dev/prod split.
dev_experiment_name = f"/Shared/{CATALOG}_refund_agent_dev"

# set_experiment creates the experiment if it doesn't exist, or activates it if it does.
dev_experiment = mlflow.set_experiment(dev_experiment_name)
dev_experiment_id = dev_experiment.experiment_id
print(f"✅ Using dev experiment: {dev_experiment_name} (ID: {dev_experiment_id})")

# Track the experiment in uc_state so `databricks bundle run cleanup` deletes it.
import sys
sys.path.append('../utils')
from uc_state import add

experiment_data = {
    "experiment_id": dev_experiment_id,
    "name": dev_experiment_name,
}
add(CATALOG, "experiments", experiment_data)
print(f"✅ Added dev experiment to UC state")

#### Production Experiment

Create the production MLflow experiment used by the Databricks App runtime for traces.

In [ ]:
import mlflow

prod_experiment_name = f"/Shared/{CATALOG}_refund_agent_prod"
prod_experiment = mlflow.set_experiment(prod_experiment_name)
prod_experiment_id = prod_experiment.experiment_id
print(f"Using prod experiment: {prod_experiment_name} (ID: {prod_experiment_id})")

import sys
sys.path.append('../utils')
from uc_state import add

add(CATALOG, "experiments", {
    "experiment_id": prod_experiment_id,
    "name": prod_experiment_name,
})
print("Added prod experiment to UC state")


#### Prompt Registry

Seed the prompt registry from the Databricks App source so prompt governance stays with the deployed app code.

In [ ]:
import os
import re
import sys

sys.path.append('../utils')
from prompt_registry import seed_prompt_history

_agent_py_path = os.path.abspath("../apps/refund-agent/agent.py")
with open(_agent_py_path) as f:
    _agent_py = f.read()

_match = re.search(r'_FALLBACK_PROMPT\s*=\s*"""(.*?)"""', _agent_py, re.DOTALL)
if not _match:
    raise RuntimeError("Could not extract _FALLBACK_PROMPT from refund app source")

_REFUND_V1 = (
    "You are a refund agent for a food delivery service. "
    "Given an order_id, decide whether to issue a refund and how much. "
    "Return a single-line JSON with `refund_usd` (float), `refund_class` "
    "(\"none\" | \"partial\" | \"full\"), and `reason` (short explanation)."
)
_REFUND_V2 = """You are RefundGPT, a CX agent responsible for refund decisions on food delivery orders.

    You can call tools to gather the information you need. Start with an `order_id`.

    Instructions:
    1. Call `order_details(order_id)` first to get event history and confirm the id is valid and the order was delivered.
    2. Figure out the delivery duration by calling `get_order_delivery_time(order_id)`.
    3. Extract the location (either directly or from the first event's body).
    4. Call `get_location_timings(location)` to get the P50/P75/P99 values.
    5. Compare actual delivery time to those percentiles.

    Refund policy (SLA-based):
       - If the order arrived AFTER the P75 delivery time: recommend a `partial` or `full` refund based on how late.
       - If the order arrived BEFORE the P75: no refund.

    Output a single-line JSON with these fields:
    - `refund_usd` (float),
    - `refund_class` (\"none\" | \"partial\" | \"full\"),
    - `reason` (short human explanation).

    You must return only the JSON. No extra text or markdown."""

_common_tags = {
    "agent": "refund",
    "stage": "refunder_agent",
    "app_name": APP_NAME,
    "uc_model": UC_MODEL_NAME,
    "consumed_via": "mlflow.genai.load_prompt at Databricks App startup",
}

seed_prompt_history(
    spark=spark,
    catalog=CATALOG,
    name="refund_system",
    historical=[
        {
            "template": _REFUND_V1,
            "commit_message": "v1: bare-bones refund decisioner, no SLA logic or tool use (demo history seed)",
            "tags": _common_tags,
        },
        {
            "template": _REFUND_V2,
            "commit_message": "v2: added tool-calling + SLA-based refund policy (P75 cutoff) (demo history seed)",
            "tags": _common_tags,
        },
    ],
    current={
        "template": _match.group(1).strip(),
        "commit_message": "v3 (production): SLA + goodwill credit path, deployed as Databricks App",
        "tags": {**_common_tags, "deployment_kind": "databricks_app"},
    },
)


#### Deploy Agent App

Create/update the Databricks App, grant the app service principal UC and Gateway access, deploy source, and register the app in uc_state.

In [ ]:
import os
import sys
import time

sys.path.append('../utils')
from agent_app_client import gateway_chat_probe
from uc_state import add

from databricks.sdk import WorkspaceClient
from databricks.sdk.service import catalog as catalog_svc
from databricks.sdk.service.apps import App, AppDeployment
from databricks.sdk.service.serving import (
    ServingEndpointAccessControlRequest,
    ServingEndpointPermissionLevel,
)

w = WorkspaceClient()
source_code_path = os.path.abspath("../apps/refund-agent")
print(f"App name: {APP_NAME}")
print(f"App source: {source_code_path}")

gateway_chat_probe(llm_model=LLM_MODEL, w=w, dbutils=dbutils)
print(f"Verified {LLM_MODEL} is queryable through Unity AI Gateway")

app_yaml_path = os.path.join(source_code_path, "app.yaml")
app_yaml_contents = f"""command:
  - python
  - start_server.py
env:
  - name: DATABRICKS_CATALOG
    value: '{CATALOG}'
  - name: LLM_MODEL
    value: '{LLM_MODEL}'
  - name: MLFLOW_EXPERIMENT_ID
    value: '{prod_experiment_id}'
  - name: MLFLOW_TRACKING_URI
    value: 'databricks'
  - name: MLFLOW_REGISTRY_URI
    value: 'databricks-uc'
"""
with open(app_yaml_path, "w") as f:
    f.write(app_yaml_contents)
print(f"Wrote app runtime config: {app_yaml_path}")

app_def = App(
    name=APP_NAME,
    description="Casper's refund decision agent served by MLflow AgentServer on Databricks Apps.",
    default_source_code_path=source_code_path,
)
try:
    w.apps.get(APP_NAME)
    print(f"App {APP_NAME} exists, updating...")
    w.apps.update(APP_NAME, app_def)
except Exception:
    print(f"Creating app {APP_NAME}...")
    w.apps.create(app_def)


def _app_state(a):
    cs = getattr(a, "compute_status", None)
    s = getattr(cs, "state", None) if cs is not None else None
    if s is None:
        s = getattr(a, "state", None)
    return getattr(s, "value", str(s)) if s is not None else ""


deadline = time.time() + 30 * 60
while True:
    current = w.apps.get(APP_NAME)
    state = _app_state(current)
    print(f"App {APP_NAME} state: {state}")
    if state in ("ACTIVE", "RUNNING", "READY"):
        app_status = current
        break
    if state in ("ERROR", "FAILED"):
        raise RuntimeError(f"App {APP_NAME} entered failure state: {state}")
    if time.time() > deadline:
        raise TimeoutError(f"App {APP_NAME} not ready after 30 minutes (last state: {state})")
    time.sleep(15)

app_sp_id = (
    getattr(app_status, "service_principal_client_id", None)
    or (app_status.as_dict() if hasattr(app_status, "as_dict") else {}).get("service_principal_client_id")
)
app_uc_principal = (
    getattr(app_status, "id", None)
    or app_sp_id
    or (app_status.as_dict() if hasattr(app_status, "as_dict") else {}).get("id")
)
assert app_sp_id, "Could not determine app service principal client ID"
assert app_uc_principal, "Could not determine app UC principal"
print(f"App SP ID: {app_sp_id}")

for full_name, securable_type, privilege in [
    (f"{CATALOG}", "CATALOG", catalog_svc.Privilege.USE_CATALOG),
    (f"{CATALOG}.ai", "SCHEMA", catalog_svc.Privilege.USE_SCHEMA),
    (f"{CATALOG}.prompts", "SCHEMA", catalog_svc.Privilege.USE_SCHEMA),
    (f"{CATALOG}.ai.get_order_details", "FUNCTION", catalog_svc.Privilege.EXECUTE),
    (f"{CATALOG}.ai.get_order_delivery_time", "FUNCTION", catalog_svc.Privilege.EXECUTE),
    (f"{CATALOG}.ai.get_location_timings", "FUNCTION", catalog_svc.Privilege.EXECUTE),
]:
    try:
        w.grants.update(
            full_name=full_name,
            securable_type=securable_type,
            changes=[
                catalog_svc.PermissionsChange(
                    add=[privilege],
                    principal=app_uc_principal,
                )
            ],
        )
        print(f"Granted {privilege} on {securable_type} {full_name}")
    except Exception as e:
        print(f"Could not grant {privilege} on {full_name} to {app_uc_principal}: {e}")

# LLM_MODEL names a Unity AI Gateway-backed endpoint. We only use the serving
# endpoint permissions API to grant CAN_QUERY on that Gateway endpoint; no LLM
# request is routed through legacy model-serving invocation routes.
llm_endpoint = None
try:
    llm_endpoint = w.serving_endpoints.get(LLM_MODEL)
except Exception:
    matches = [ep for ep in w.serving_endpoints.list() if ep.name == LLM_MODEL]
    if matches:
        llm_endpoint = matches[0]
if llm_endpoint is None or not getattr(llm_endpoint, "id", None):
    raise RuntimeError(f"Could not resolve Gateway endpoint {LLM_MODEL} for permission grant")

w.serving_endpoints.update_permissions(
    serving_endpoint_id=llm_endpoint.id,
    access_control_list=[
        ServingEndpointAccessControlRequest(
            service_principal_name=app_sp_id,
            permission_level=ServingEndpointPermissionLevel.CAN_QUERY,
        )
    ],
)
print(f"Granted CAN_QUERY on Gateway endpoint {LLM_MODEL} to app SP {app_sp_id}")

try:
    w.api_client.do(
        "PATCH",
        f"/api/2.0/permissions/apps/{APP_NAME}",
        body={"access_control_list": [{"group_name": "account users", "permission_level": "CAN_USE"}]},
    )
    print("Granted CAN_USE on app to account users for notebook/job smoke tests")
except Exception as e:
    print(f"Could not grant account users CAN_USE on app {APP_NAME}: {e}")

add(CATALOG, "apps", {
    "name": APP_NAME,
    "url": getattr(app_status, "url", ""),
    "service_principal_client_id": app_sp_id,
    "oauth2_app_client_id": getattr(app_status, "oauth2_app_client_id", ""),
    "agent": 'refund',
})
print("Registered app in UC state")

deployment = w.apps.deploy(
    app_name=app_status.name,
    app_deployment=AppDeployment(source_code_path=source_code_path),
)


def _deploy_state(d):
    st = getattr(d, "status", None)
    s = getattr(st, "state", None) if st is not None else None
    return getattr(s, "value", str(s)) if s is not None else ""


deadline = time.time() + 30 * 60
while True:
    current_dep = w.apps.get_deployment(app_name=app_status.name, deployment_id=deployment.deployment_id)
    state = _deploy_state(current_dep)
    print(f"Deployment state: {state}")
    if state == "SUCCEEDED":
        deployment_status = current_dep
        break
    if state in ("FAILED", "STOPPED"):
        raise RuntimeError(f"Deployment failed for {app_status.name}: state={state}")
    if time.time() > deadline:
        raise TimeoutError(f"Deployment for {app_status.name} not ready after 30 minutes (last state: {state})")
    time.sleep(10)

print(f"Refund agent app deployed: {getattr(app_status, 'url', '')}")
display(deployment_status)


#### Production Monitoring

Register MLflow scorers on the prod experiment so every live request that
hits the refund agent is automatically scored.  Mirrors the in-stage
pattern in `stages/complaint_agent.ipynb` and the helper in
`demos/operational-dashboard-demo/evaluation.ipynb`.

Scorer set (4, all at 100% sampling):

- `safety` — built-in `Safety()` LLM judge for harmful or inappropriate content
- `relevance_to_query` — built-in `RelevanceToQuery()` LLM judge — does the answer address the question
- `operational_quality` — generic `Guidelines` — concrete data, not a hedge
- `refund_policy_compliance` — refund-specific `Guidelines` — recommendation matches the policy

In [ ]:
from mlflow.genai.scorers import (
    Safety,
    RelevanceToQuery,
    Guidelines,
    ScorerSamplingConfig,
    list_scorers,
)


def _register_scorer(scorer_obj, name: str, sample_rate: float = 1.0):
    """Idempotent register-or-restart, scoped to the prod experiment.

    Re-running the stage hits the same code path; calling .register() on
    an already-registered scorer raises ValueError, so we look it up first
    and just .start() it instead.  Same shape as
    stages/complaint_agent.ipynb and demos/.../evaluation.ipynb.
    """
    existing = {s.name: s for s in list_scorers(experiment_id=prod_experiment_id)}
    sampling = ScorerSamplingConfig(sample_rate=sample_rate)
    if name in existing:
        existing[name].start(sampling_config=sampling)
        print(f"  ↺ {name} — restarted at {sample_rate:.0%} sample rate")
        return existing[name]
    registered = scorer_obj.register(name=name, experiment_id=prod_experiment_id)
    registered.start(sampling_config=sampling)
    print(f"  ✅ {name} — registered + started at {sample_rate:.0%} sample rate")
    return registered


# Baseline (every agent in the bundle gets these three).
_register_scorer(Safety(),           name="safety",             sample_rate=1.0)
_register_scorer(RelevanceToQuery(), name="relevance_to_query", sample_rate=1.0)
_register_scorer(
    Guidelines(
        name="operational_quality",
        guidelines=(
            "The response must include specific data points such as numbers, "
            "percentages, dates, order IDs, or named locations. "
            "The response must not be a generic hedge or refusal "
            "(e.g. 'I don't have access'). "
            "The response must directly answer the question asked."
        ),
    ),
    name="operational_quality",
    sample_rate=1.0,
)

# Domain — refund-specific policy compliance.
_register_scorer(
    Guidelines(
        name="refund_policy_compliance",
        guidelines=[
            "Recommendations must be one of: no refund, partial refund, or full refund — never anything outside this set.",
            "If a refund is recommended, the response must cite the order_id it applies to.",
            "If the order is older than the refund window, the response must explicitly mention that the refund window has expired.",
            "The response must reference the specific reason from the order data (e.g. late delivery, missing item, food quality).",
        ],
    ),
    name="refund_policy_compliance",
    sample_rate=1.0,
)

print("✅ Production monitoring enabled — 4 scorers active at 100% sampling")